# Section 3: Traffic anomaly detection

Clean one CIC-IDS2018 flow file and compare Isolation Forest with an autoencoder. Both fit benign traffic only; labelled validation data selects an operational alert threshold.

This revision moves missingness selection behind the split, encodes protocol/port semantics, adds input EDA and overlap checks, imposes a validation false-alert budget, packages preprocessing, and records provenance. Outputs were cleared; rerun top to bottom.

## Setup

The working directory changes only in Colab. Local runs locate the repository root automatically. The run summary records package versions and the dataset hash.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    os.chdir("/content")

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import random
import shutil
import sys
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

### Experiment settings

Keep the seed, epoch count and output folders together. Colab uses a separate remote workspace. Seeds help repeatability, but they do not guarantee identical results across environments.

In [ ]:
seed = 42
epochs = 12
max_validation_fpr = 0.05
run_id = datetime.now(timezone.utc).strftime("s03-%Y%m%dT%H%M%SZ")

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "notebooks").is_dir() and (candidate / "docs").is_dir():
            return candidate
    return Path.cwd()

in_colab = "google.colab" in sys.modules
root = Path("/content/section_03_workspace") if in_colab else find_project_root()
data_file = root / "data/raw/cic-ids2018/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv"
processed_dir = root / "data/processed/section_03"
model_dir = root / "models/section_03"
results_dir = root / "reports/section_03"
for directory in (data_file.parent, processed_dir, model_dir, results_dir / "metrics", results_dir / "figures"):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Load the flow data

Download the unclean flow-feature CSV for 1 March 2018, unless it already exists locally. These are extracted flow measurements, not raw packets.

In [ ]:
data_url = "https://cse-cic-ids2018.s3.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv"
if not data_file.exists():
    request = urllib.request.Request(data_url, headers={"User-Agent": "COMP70049-assignment"})
    with urllib.request.urlopen(request) as response, data_file.open("wb") as output:
        shutil.copyfileobj(response, output)
dataset_hash = hashlib.sha256(data_file.read_bytes()).hexdigest()

## Clean and inspect the data

Convert flow measurements to numeric values, derive hour/weekday, and represent protocol plus destination-port role as categorical features rather than continuous magnitudes. The raw label is never an input. Missingness-based feature removal happens only after the benign training split is defined.

In [ ]:
# Choose the flow measurements used by both models, plus two time features.
traffic_cols = [
    "Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts", "TotLen Bwd Pkts",
    "Fwd Pkt Len Max", "Fwd Pkt Len Min", "Fwd Pkt Len Mean", "Fwd Pkt Len Std", "Bwd Pkt Len Max", "Bwd Pkt Len Min", "Bwd Pkt Len Mean", "Bwd Pkt Len Std",
    "Flow Byts/s", "Flow Pkts/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Tot", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min", "Bwd IAT Tot", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd Pkts/s", "Bwd Pkts/s", "Pkt Len Min", "Pkt Len Max", "Pkt Len Mean", "Pkt Len Std", "Pkt Len Var",
    "FIN Flag Cnt", "SYN Flag Cnt", "RST Flag Cnt", "PSH Flag Cnt", "ACK Flag Cnt", "URG Flag Cnt", "Down/Up Ratio", "Pkt Size Avg",
    "Fwd Seg Size Avg", "Bwd Seg Size Avg", "Subflow Fwd Pkts", "Subflow Fwd Byts", "Subflow Bwd Pkts", "Subflow Bwd Byts",
    "Init Fwd Win Byts", "Init Bwd Win Byts", "Fwd Act Data Pkts", "Fwd Seg Size Min",
    "Active Mean", "Active Std", "Active Max", "Active Min", "Idle Mean", "Idle Std", "Idle Max", "Idle Min", "hour", "day_of_week",
]

In [ ]:
raw = pd.read_csv(data_file, low_memory=False)
# Count the original rows, missing values and duplicates before cleaning.
raw_profile = {
    "records": len(raw), "columns": raw.shape[1],
    "duplicate_records": int(raw.duplicated().sum()),
    "native_missing_values": int(raw.isna().sum().sum()),
}

df = raw.copy()
# Tidy the headers and remove unnamed export columns.
df.columns = [str(column).replace("\ufeff", "").strip() for column in df.columns]
df = df.drop(columns=[column for column in df.columns if column.lower().startswith("unnamed:")])
labels = df["Label"].astype("string").str.strip()
# Remove rows with no label and header lines repeated inside the CSV.
missing_labels = labels.isna() | labels.eq("")
embedded_headers = labels.str.casefold().eq("label")
df = df.loc[~(missing_labels | embedded_headers)].copy()
# Count and remove the duplicate rows left after removing headers.
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
labels = df["Label"].astype("string").str.strip()

# Extract hour and weekday from the recorded flow time.
timestamps = pd.to_datetime(
    df["Timestamp"].astype("string").str.strip(), errors="coerce", format="mixed", dayfirst=True,
)
df["hour"] = timestamps.dt.hour
df["day_of_week"] = timestamps.dt.dayofweek

### Convert the measurements

Turn malformed numbers and infinities into missing values for training-fitted imputation. Protocol is categorical; destination ports become common-service or IANA-range categories. No data-dependent feature is removed at this stage.

In [ ]:
numeric = pd.DataFrame(index=df.index)
infinities = 0
coercions = 0
for column in traffic_cols:
    original = df[column]
    converted = pd.to_numeric(original, errors="coerce")
    coercions += int((original.notna() & original.astype("string").str.strip().ne("") & converted.isna()).sum())
    infinities += int(np.isinf(converted.to_numpy(float)).sum())
    numeric[column] = converted.replace([np.inf, -np.inf], np.nan)

protocol_values = numeric.pop("Protocol")
port_values = numeric.pop("Dst Port")
numeric_feature_candidates = numeric.columns.tolist()
common_ports = {20: "ftp-data", 21: "ftp", 22: "ssh", 23: "telnet", 25: "smtp",
                53: "dns", 80: "http", 110: "pop3", 143: "imap", 443: "https",
                445: "smb", 3389: "rdp"}

def protocol_category(value):
    if pd.isna(value):
        return pd.NA
    return {1: "icmp", 6: "tcp", 17: "udp"}.get(int(value), "other")

def port_category(value):
    if pd.isna(value):
        return pd.NA
    port = int(value)
    if port in common_ports:
        return f"service_{common_ports[port]}"
    if 0 <= port <= 1023:
        return "other_system"
    if port <= 49151:
        return "registered"
    if port <= 65535:
        return "dynamic"
    return "invalid"

cleaned = numeric.copy()
cleaned["protocol_category"] = protocol_values.map(protocol_category).astype(object).where(protocol_values.notna(), np.nan)
cleaned["destination_port_category"] = port_values.map(port_category).astype(object).where(port_values.notna(), np.nan)
categorical_feature_cols = ["protocol_category", "destination_port_category"]
cleaned["attack_label"] = labels.to_numpy(str)
cleaned["label"] = (~labels.str.casefold().eq("benign")).astype(int).to_numpy()

### Review the cleaning counts

Save the before-and-after counts for the report. Repeated headers are removed before counting the remaining duplicates, so those counts need not match the raw duplicate total.

In [ ]:
cleaning_report = {
    "records_after_cleaning": len(cleaned),
    "rows_without_label_removed": int(missing_labels.sum()),
    "embedded_header_rows_removed": int(embedded_headers.sum()),
    "duplicate_records_removed": duplicates_removed,
    "infinite_values_replaced_with_missing": infinities,
    "non_numeric_values_coerced_to_missing": coercions,
    "invalid_timestamps": int(timestamps.isna().sum()),
    "remaining_missing_values_before_training_fitted_selection": int(cleaned.isna().sum().sum()),
    "benign_records": int((cleaned["label"] == 0).sum()),
    "anomaly_records": int((cleaned["label"] == 1).sum()),
}
display(pd.Series({**raw_profile, **cleaning_report}).to_frame("value"))

## Split the flows and fit the feature boundary

Sample 120,000 benign flows and 18,000 attacks. Define the benign training pool first, then use **only that pool** to remove features above 40% missingness. Validation/test each contain 24,000 benign and 9,000 attack flows. Input summaries and exact representation-overlap checks describe the resulting experiment.

In [ ]:
benign = cleaned[cleaned["label"] == 0].sample(n=120000, random_state=seed).reset_index(drop=True)
anomalies = cleaned[cleaned["label"] == 1].sample(n=18000, random_state=seed + 1).reset_index(drop=True)
train = benign.iloc[:72000].copy()
val = pd.concat([benign.iloc[72000:96000], anomalies.iloc[:9000]], ignore_index=True)
test = pd.concat([benign.iloc[96000:], anomalies.iloc[9000:]], ignore_index=True)
val = val.sample(frac=1, random_state=seed + 2).reset_index(drop=True)
test = test.sample(frac=1, random_state=seed + 3).reset_index(drop=True)

# This selection is fitted on benign training records only.
dropped_features = train[numeric_feature_candidates].columns[
    train[numeric_feature_candidates].isna().mean() > 0.4
].tolist()
numeric_feature_cols = [column for column in numeric_feature_candidates if column not in dropped_features]
feature_cols = [*numeric_feature_cols, *categorical_feature_cols]
cleaning_report["features_dropped_for_training_missingness"] = dropped_features
(processed_dir / "data-quality-report.json").write_text(
    json.dumps({"raw_profile": raw_profile, "cleaning_report": cleaning_report}, indent=2)
)

representation_hashes = {
    name: set(pd.util.hash_pandas_object(part[feature_cols], index=False).astype(str))
    for name, part in (("train", train), ("validation", val), ("test", test))
}
overlap = {
    "train_validation": len(representation_hashes["train"] & representation_hashes["validation"]),
    "train_test": len(representation_hashes["train"] & representation_hashes["test"]),
    "validation_test": len(representation_hashes["validation"] & representation_hashes["test"]),
}
(processed_dir / "representation-overlap.json").write_text(json.dumps(overlap, indent=2))

skewness = train[numeric_feature_cols].skew(numeric_only=True).sort_values(key=np.abs, ascending=False)
skewness.rename("skewness").to_csv(processed_dir / "numeric-skewness.csv")
correlations = train[numeric_feature_cols].corr(numeric_only=True)
correlation_pairs = correlations.where(np.triu(np.ones(correlations.shape), 1).astype(bool)).stack()
correlation_pairs.reindex(correlation_pairs.abs().sort_values(ascending=False).index).head(50).rename(
    "correlation"
).to_csv(processed_dir / "top-numeric-correlations.csv")
plot_cols = skewness.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for column, axis in zip(plot_cols, axes.flat):
    values = train[column].replace([np.inf, -np.inf], np.nan).dropna()
    axis.hist(np.sign(values) * np.log1p(np.abs(values)), bins=50)
    axis.set_title(column)
fig.tight_layout()
fig.savefig(results_dir / "figures/input-feature-distributions.png", dpi=180)
plt.show()

display(pd.DataFrame([
    {"split": name, "records": len(part), "benign": int((part["label"] == 0).sum()),
     "anomaly": int((part["label"] == 1).sum())}
    for name, part in (("train", train), ("validation", val), ("test", test))
]))
display(pd.Series(overlap).to_frame("exact matching representations"))

## Prepare the features

Fit median imputation with missingness indicators and scaling on numeric benign-training features. Fit most-frequent imputation plus one-hot encoding on semantic protocol/port categories. Remove constant transformed columns using training data, then apply the pipeline unchanged to validation and test flows.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = Pipeline([
    ("columns", ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_feature_cols),
        ("categorical", categorical_pipeline, categorical_feature_cols),
    ], verbose_feature_names_out=False)),
    ("variance", VarianceThreshold()),
])
X_train = preprocessor.fit_transform(train[feature_cols]).astype(np.float32)
X_val = preprocessor.transform(val[feature_cols]).astype(np.float32)
X_test = preprocessor.transform(test[feature_cols]).astype(np.float32)
y_val = val["label"].to_numpy(np.int64)
y_test = test["label"].to_numpy(np.int64)

feature_names = preprocessor.named_steps["columns"].get_feature_names_out()
feature_names = feature_names[preprocessor.named_steps["variance"].get_support()].tolist()
(processed_dir / "selected-features.json").write_text(json.dumps(feature_names, indent=2))
joblib.dump(preprocessor, model_dir / "feature-preprocessor.joblib")
display(pd.DataFrame({"split": ["train", "validation", "test"],
                      "records": [len(X_train), len(X_val), len(X_test)],
                      "features": [X_train.shape[1]] * 3}))

## Choose the alert threshold

Choose the validation ROC point with the highest attack recall while keeping false positives at or below 5%. This policy is declared before test evaluation and prioritizes an operational alert-volume constraint over maximising validation F1.

In [ ]:
def select_threshold(labels, scores, max_fpr=max_validation_fpr):
    labels, scores = np.asarray(labels), np.asarray(scores)
    fpr, tpr, thresholds = roc_curve(labels, scores)
    eligible = np.flatnonzero(fpr <= max_fpr)
    best_index = max(eligible, key=lambda index: (tpr[index], -fpr[index]))
    threshold = float(thresholds[best_index])
    validation_preds = scores >= threshold
    tn, fp, fn, tp = confusion_matrix(labels, validation_preds, labels=[0, 1]).ravel()
    return threshold, {
        "policy": "maximum validation recall subject to false-positive-rate budget",
        "maximum_validation_fpr": max_fpr,
        "observed_validation_fpr": float(fp / max(fp + tn, 1)),
        "observed_validation_recall": float(tp / max(tp + fn, 1)),
    }

### Evaluate the alerts

Convert scores to alerts using the chosen threshold. Report how many attacks are caught, how many benign flows are flagged and how the score distributions overlap.

In [ ]:
def show_results(name, labels, scores, threshold, threshold_selection, filename):
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    metrics = {
        "run_id": run_id,
        "threshold": float(threshold),
        "threshold_selection": threshold_selection,
        "true_positive_rate": float(tp / (tp + fn)),
        "false_positive_rate": float(fp / (fp + tn)),
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "recall": float(recall_score(labels, preds, zero_division=0)),
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "average_precision": float(average_precision_score(labels, scores)),
        "roc_auc": float(roc_auc_score(labels, scores)),
        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
    }
    (results_dir / "metrics" / f"{filename}.json").write_text(json.dumps(metrics, indent=2))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ConfusionMatrixDisplay.from_predictions(
        labels, preds, display_labels=["Benign", "Anomaly"], cmap="Blues", colorbar=False, ax=axes[0],
    )
    precision, recall, _ = precision_recall_curve(labels, scores)
    axes[1].plot(recall, precision)
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve")
    axes[2].hist(scores[labels == 0], bins=60, alpha=0.65, label="Benign")
    axes[2].hist(scores[labels == 1], bins=60, alpha=0.65, label="Anomaly")
    axes[2].axvline(threshold, color="black", linestyle="--")
    axes[2].set_title("Score distribution")
    axes[2].legend()
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(results_dir / "figures" / f"{filename}-evaluation.png", dpi=180)
    plt.show()
    return metrics

## Isolation Forest

Fit random isolation trees to benign flows. We negate the normality scores so larger values mean more unusual traffic, then choose a validation F1 threshold instead of using the model's default decision boundary.

In [ ]:
iso_model = IsolationForest(
    n_estimators=200, max_samples=10000, contamination="auto", random_state=seed, n_jobs=1,
)
iso_model.fit(X_train)
iso_val_scores = -iso_model.score_samples(X_val)
iso_threshold, iso_threshold_selection = select_threshold(y_val, iso_val_scores)
iso_scores = -iso_model.score_samples(X_test)
iso_metrics = show_results(
    "Isolation Forest", y_test, iso_scores, iso_threshold, iso_threshold_selection, "isolation-forest",
)
joblib.dump({"model": iso_model, "preprocessor": preprocessor,
             "threshold": iso_threshold, "feature_names": feature_names, "run_id": run_id},
            model_dir / "isolation-forest.joblib")
display(pd.Series(iso_metrics).drop("confusion_matrix").to_frame("value"))

## Autoencoder

Learn to reconstruct benign traffic through a 12-unit bottleneck. A larger reconstruction error is treated as a stronger anomaly score. The last layer is linear because standardized inputs can be negative.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, feature_count):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_count, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            # Compress to twelve values before reconstructing the input features.
            nn.Linear(32, 12),
            nn.Linear(12, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, feature_count),
        )

    def forward(self, values):
        return self.network(values)

def make_loader(features, shuffle=False):
    # Wrap the feature array so the data loader can return batches.
    dataset = TensorDataset(torch.from_numpy(features))
    return DataLoader(dataset, batch_size=1024, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed) if shuffle else None)

### Set up reconstruction training

Use batches of benign flows and minimize mean squared reconstruction error. Only benign validation flows select the checkpoint; the full labelled validation set is used later to choose the alert threshold.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ae_model = Autoencoder(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(ae_model.parameters(), lr=0.001, weight_decay=0.00001)
loss_fn = nn.MSELoss()
train_loader = make_loader(X_train, True)
# Use only benign validation flows to judge reconstruction quality.
X_val_normal = X_val[y_val == 0]

def reconstruction_error(features):
    # Turn off dropout and gradient tracking while measuring reconstruction error.
    ae_model.eval()
    scores = []
    with torch.no_grad():
        for (batch,) in make_loader(features):
            batch = batch.to(device)
            # Average the squared feature errors to get one anomaly score per flow.
            scores.extend(torch.mean((ae_model(batch) - batch) ** 2, dim=1).cpu().numpy())
    return np.asarray(scores)

### Train and evaluate

Run all twelve epochs and restore the lowest benign-validation-loss checkpoint. Plot both losses, select the alert threshold on validation data, then evaluate the test flows.

The autoencoder checkpoint stores weights, threshold and feature names, but not the fitted preprocessor. Keep that pipeline too; it is included in the Isolation Forest artifact.

In [ ]:
history = []
best_val_loss = float("inf")
for epoch in range(1, epochs + 1):
    ae_model.train()
    total_loss = 0
    for (batch,) in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = loss_fn(ae_model(batch), batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch)

    val_loss = reconstruction_error(X_val_normal).mean()
    history.append({"epoch": epoch, "training_loss": total_loss / len(X_train),
                    "normal_validation_loss": val_loss})
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        chosen_epoch = epoch
        best_state = {name: value.detach().cpu().clone() for name, value in ae_model.state_dict().items()}

ae_model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_df.to_csv(results_dir / "metrics/autoencoder-training-history.csv", index=False)
display(history_df)
history_df.set_index("epoch").plot(figsize=(7, 4), title="Autoencoder training history")
plt.tight_layout()
plt.savefig(results_dir / "figures/autoencoder-training-history.png", dpi=180)
plt.show()

ae_val_scores = reconstruction_error(X_val)
ae_threshold, ae_threshold_selection = select_threshold(y_val, ae_val_scores)
ae_scores = reconstruction_error(X_test)
ae_metrics = show_results(
    "Autoencoder", y_test, ae_scores, ae_threshold, ae_threshold_selection, "autoencoder",
)
ae_metrics.update({"best_normal_validation_loss": float(best_val_loss), "chosen_epoch": chosen_epoch,
                   "epochs": epochs, "device": str(device)})
(results_dir / "metrics/autoencoder.json").write_text(json.dumps(ae_metrics, indent=2))
torch.save({"model_state": best_state, "threshold": ae_threshold,
            "feature_names": feature_names, "run_id": run_id}, model_dir / "autoencoder.pt")
display(pd.Series(ae_metrics).drop("confusion_matrix").to_frame("value"))

## Compare the models

Compare recall with false-alert rate under the predeclared validation budget. Save metrics, histories, the quality report, feature list, fitted preprocessor, models and environment metadata under one run identifier.

In [ ]:
metric_names = ["true_positive_rate", "false_positive_rate", "precision", "f1", "average_precision", "roc_auc"]
comparison = pd.DataFrame([
    {"model": "Isolation Forest", **{metric: iso_metrics[metric] for metric in metric_names}},
    {"model": "Autoencoder", **{metric: ae_metrics[metric] for metric in metric_names}},
])
comparison.to_csv(results_dir / "model-comparison.csv", index=False)
packages = ["joblib", "matplotlib", "numpy", "pandas", "scikit-learn", "torch"]
(results_dir / "run-summary.json").write_text(json.dumps({
    "run_id": run_id, "created_utc": datetime.now(timezone.utc).isoformat(), "seed": seed,
    "dataset_sha256": dataset_hash,
    "train_records": len(train), "validation_records": len(val), "test_records": len(test),
    "features": X_train.shape[1], "autoencoder_epochs": epochs,
    "autoencoder_chosen_epoch": chosen_epoch, "maximum_validation_fpr": max_validation_fpr,
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {name: importlib.metadata.version(name) for name in packages},
}, indent=2))
display(comparison.style.format({metric: "{:.4f}" for metric in metric_names}))

if in_colab:
    export_dir = root / "section_03_export"
    shutil.copytree(results_dir, export_dir / "reports", dirs_exist_ok=True)
    shutil.copytree(model_dir, export_dir / "models", dirs_exist_ok=True)
    shutil.copytree(processed_dir, export_dir / "processed", dirs_exist_ok=True)
    shutil.make_archive("/content/section_03_results", "zip", root_dir=export_dir)

## What to take from the results

- The 5% validation false-alert budget makes the operational trade-off explicit; report any loss of recall alongside the lower alert volume.
- Labels select benign training/validation pools, define evaluation splits and set thresholds. They do not enter either fitting loss.
- This remains a random split from one day, not a future-day or zero-day test. Exact representation checks help identify overlap but do not establish attack-family independence.